# 03 · Выравнивание по предпочтениям

**Чего не умеет SFT.** Он видит только эталонный ответ и никогда не наблюдает, чего говорить не следует. Градиент поднимает вероятность эталона, но не опускает вероятность правдоподобной альтернативы.

В `prefs.jsonl` отвергнутые ответы собраны так, что звучат убедительно: гладкая канцелярская гипотеза, технически корректный совет не по делу. Именно на таких SFT не помогает.

Вывод DPO и остальных — `books/03-alignment.pdf`.

In [ ]:
from common import MODEL_ID, SYSTEM, DATA, RUNS, demo_answers, show, side_by_side, policy_suite, fmt, read_raw

import torch
from datasets import Dataset
from transformers import AutoModelForImageTextToText, AutoProcessor
from peft import LoraConfig
from vlmkit import memory_report, evaluate as ev
from vlmkit.compat import alignment_trainer, available_alignment, supported

print("доступно в вашем trl:", available_alignment())

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map={"": 0},
    attn_implementation="sdpa", trust_remote_code=True,
)
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True, max_pixels=1003520)
print(memory_report())

## До

In [ ]:
suite = policy_suite()
before = demo_answers(model, processor, system=SYSTEM)
before_metrics = ev.run(model, processor, suite)
show(before, "ДО ВЫРАВНИВАНИЯ")
print("\nметрики:", fmt(before_metrics))

## Пары

Посмотрите на одну пару глазами. Отвергнутый ответ — не мусор, а грамотный текст, нарушающий одно правило.

In [ ]:
pairs_raw = read_raw("prefs.jsonl")
pairs = Dataset.from_list(pairs_raw)
print(pairs)

p = pairs_raw[0]
print(f"\nЗАПРОС:    {p['prompt']}")
print(f"ВЫБРАННЫЙ: {p['chosen'][:200]}")
print(f"ОТВЕРГНУТЫЙ: {p['rejected'][:200]}")

## Тренер

`alignment_trainer` находит, каким классом ваша версия trl реализует метод — в trl 1.x отдельные `ORPOTrainer` и `CPOTrainer` слиты в `DPOTrainer` с разными `loss_type`. Сам вызов ниже обычный.

Скорость обучения на порядок ниже, чем в SFT: выравнивание правит уже обученное поведение, и большой шаг его разрушает.

`beta` означает разное: у DPO — сила KL-штрафа (0.1), у SimPO — масштаб награды (2.0–2.5). Меняйте вместе с методом.

In [ ]:
METHOD = "orpo"          # или simpo / dpo / kto — см. available_alignment() выше
Config, TrainerCls, extra = alignment_trainer(METHOD)
print(f"{METHOD} → {TrainerCls.__name__} {extra}")

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=r"^(?!.*(visual|vision)).*(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)$",
    use_rslora=True, bias="none", task_type="CAUSAL_LM",
)

args = Config(**supported(Config, {
    **extra,
    "output_dir": str(RUNS / METHOD),
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "num_train_epochs": 3,
    "learning_rate": 5e-6,
    "beta": 2.0 if METHOD == "simpo" else 0.1,
    "bf16": True,
    "max_length": 2048,
    "max_prompt_length": 1024,
    "gradient_checkpointing": True,
    "gradient_checkpointing_kwargs": {"use_reentrant": False},
    "logging_steps": 5,
    "save_strategy": "no",
    "report_to": [],
    "remove_unused_columns": False,
}))

trainer = TrainerCls(
    model=model, args=args, train_dataset=pairs,
    processing_class=processor,
    peft_config=lora,       # выравнивание поверх адаптера: результат отключаемый
)
trainer.train()
model = trainer.model

## После

In [ ]:
model.eval()
after = demo_answers(model, processor, system=SYSTEM)
after_metrics = ev.run(model, processor, suite)

show(after, f"ПОСЛЕ {METHOD.upper()}")
side_by_side(before, after, detector=lambda t: "?" in t)
print(f"\nдо:    {fmt(before_metrics)}")
print(f"после: {fmt(after_metrics)}")

model.save_pretrained(str(RUNS / METHOD))

## Что должно отличаться от SFT

SFT и выравнивание учат разному, и это видно на группе `answer`.

SFT на `policy.jsonl` учил модель уточнять — и она склонна уточнять везде, в том числе на справочных вопросах. Выравнивание видело пары, где `chosen` — прямой ответ, а `rejected` — лишнее уточнение. Оно учило **различать**, когда уточнять надо, а когда нет.

Если после выравнивания ложные срабатывания ниже, чем после SFT при том же попадании — это и есть тот эффект, ради которого нужен второй этап.